# **Projeto IA Dália - Exploração do RAG**

## **Objetivo**
Desenvolver e avaliar um mecanismo de recuperação de conhecimento
(Retrieval-Augmented Generation — RAG) para a Dália AI.

## **Etapas**
1. Carregamento e inspeção da base de conhecimento.
2. Pré-processamento textual.
3. Vetorização com TF-IDF.
4. Recuperação por similaridade de cosseno.
5. Testes com perguntas de diferentes domínios.
6. Avaliação da relevância dos resultados.

## **Fonte de dados**
`data/knowledge/v2/dalia_knowledge_v2_revisada.jsonl`

### **Importando Bibliotecas e Materiais:**

In [1]:
import json
import re
import unicodedata

from pathlib import Path
from collections import Counter

import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


### **Localização e Carregamento da Base:**

In [2]:
PROJECT_DIR = Path.cwd().resolve()

# Caso o notebook seja executado a partir de outra pasta,
# procuramos a raiz do projeto.
if not (PROJECT_DIR / "data/knowledge/v2").exists():
    PROJECT_DIR = PROJECT_DIR.parent.parent

KNOWLEDGE_PATH = (
    PROJECT_DIR
    / "data"
    / "knowledge"
    / "v2"
    / "dalia_knowledge_v2_revisada.jsonl"
)

if not KNOWLEDGE_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo de conhecimento não encontrado: {KNOWLEDGE_PATH}"
    )

with KNOWLEDGE_PATH.open("r", encoding="utf-8") as arquivo:
    knowledge_base = [
        json.loads(linha)
        for linha in arquivo
        if linha.strip()
    ]

print(f"Projeto: {PROJECT_DIR}")
print(f"Arquivo: {KNOWLEDGE_PATH}")
print(f"Registros carregados: {len(knowledge_base)}")

Projeto: /media/natali/27a81e87-8b76-410b-906a-586a76f2704c/outros/outros projetos/daliaIA
Arquivo: /media/natali/27a81e87-8b76-410b-906a-586a76f2704c/outros/outros projetos/daliaIA/data/knowledge/v2/dalia_knowledge_v2_revisada.jsonl
Registros carregados: 47


### **Inspeção da Base de Conhecimento:**

In [3]:
# Transformar os registros em um DataFrame
df_knowledge = pd.DataFrame(knowledge_base)

# Visualizar os primeiros registros
display(
    df_knowledge[
        ["id", "domain", "topic", "title", "urgency"]
    ].head(10)
)

print(f"\nTotal de documentos: {len(df_knowledge)}")
print(f"Total de domínios: {df_knowledge['domain'].nunique()}")
print(f"Total de tópicos: {df_knowledge['topic'].nunique()}")

,id,domain,topic,title,urgency
0,GEST_001,gestacional,inicio_pre_natal,Início e continuidade do pré-natal,rotina
1,GEST_002,gestacional,sinais_alerta_gestacao,Sinais de alerta durante a gestação,emergencia
2,GEST_003,gestacional,classificacao_risco,Risco habitual e alto risco gestacional,informacao_clinica
3,GEST_004,gestacional,puerperio,Puerpério e acompanhamento após o parto,informacao_clinica
4,MEN_001,menstrual,ciclo_menstrual,Como funciona o ciclo menstrual,educacao
5,MEN_002,menstrual,sinais_menstruais_alerta,Quando alterações menstruais merecem avaliação,avaliacao_ou_urgencia_conforme_sintomas
6,MEN_003,menstrual,dismenorreia,Cólicas menstruais,avaliacao
7,MEN_004,menstrual,endometriose,Endometriose: informação e procura de cuidado,avaliacao
8,MEN_005,menstrual,sop,Síndrome dos ovários policísticos (SOP),avaliacao
9,MENT_001,saude_mental,sofrimento_emocional,Acolhimento diante de sofrimento emocional,acolhimento



Total de documentos: 47
Total de domínios: 6
Total de tópicos: 47


### **Distribuição por Domínio e Urgência:**

In [4]:
print("========== DOCUMENTOS POR DOMÍNIO ==========")

display(
    df_knowledge["domain"]
    .value_counts()
    .rename_axis("Domínio")
    .to_frame("Quantidade")
)

print("\n========== DOCUMENTOS POR URGÊNCIA ==========")

display(
    df_knowledge["urgency"]
    .value_counts()
    .rename_axis("Urgência")
    .to_frame("Quantidade")
)

========== DOCUMENTOS POR DOMÍNIO ==========


,Quantidade
Domínio,
gestacional,11
menstrual,9
violencia,8
saude_sexual_reprodutiva,8
saude_mental,7
seguranca_clinica,4



========== DOCUMENTOS POR URGÊNCIA ==========


,Quantidade
Urgência,
avaliacao,15
emergencia,6
rotina,6
governanca,3
sempre_aplicavel,3
seguranca,3
informacao_clinica,2
educacao,2
avaliacao_ou_urgencia_conforme_sintomas,1


### **Análise do Tamanho dos Documentos:**

In [5]:
df_knowledge["content_length"] = (
    df_knowledge["content"].str.len()
)

print("========== TAMANHO DOS DOCUMENTOS ==========")

display(
    df_knowledge["content_length"].describe()
)

print("\nMenor documento:")
display(
    df_knowledge.loc[
        df_knowledge["content_length"].idxmin(),
        ["id", "title", "content_length"]
    ]
)

print("\nMaior documento:")
display(
    df_knowledge.loc[
        df_knowledge["content_length"].idxmax(),
        ["id", "title", "content_length"]
    ]
)

========== TAMANHO DOS DOCUMENTOS ==========


count      47.000000
mean      675.489362
std       189.154971
min       430.000000
25%       506.000000
50%       661.000000
75%       810.000000
max      1131.000000
Name: content_length, dtype: float64


Menor documento:


id                                                MEN_005
title             Síndrome dos ovários policísticos (SOP)
content_length                                        430
Name: 8, dtype: object


Maior documento:


id                                                          SEX_003
title             Infecções sexualmente transmissíveis durante a...
content_length                                                 1131
Name: 27, dtype: object

### **Construção dos Textos de Busca:**

In [6]:
def build_search_text(record):
    keywords = " ".join(record["keywords"])

    return " ".join([
        record["title"],
        record["topic"].replace("_", " "),
        keywords,
        record["content"]
    ])


df_knowledge["search_text"] = [
    build_search_text(record)
    for record in knowledge_base
]

print("Textos de busca construídos!")
print(f"Total: {len(df_knowledge['search_text'])}")

# Inspecionar um documento
print("\n========== EXEMPLO DE TEXTO ==========")
print(df_knowledge.loc[0, "search_text"])

Textos de busca construídos!
Total: 47

========== EXEMPLO DE TEXTO ==========
Início e continuidade do pré-natal inicio pre natal gravidez gestação pré-natal primeira consulta UBS 12 semanas A pessoa que descobre ou suspeita de gravidez deve procurar uma Unidade Básica de Saúde o quanto antes para iniciar o pré-natal, idealmente até a 12ª semana. O acompanhamento permite avaliar a saúde da gestante, acompanhar o desenvolvimento da gestação, solicitar exames e identificar necessidades de cuidado. O calendário e a frequência das consultas podem variar conforme avaliação clínica, protocolo local e necessidades individuais. O Dália deve incentivar o acompanhamento profissional e não substituir consultas.


### **Preparação do TF-IDF:**

- TF: frequência de um termo dentro de um documento.

- IDF: o quanto esse termo é específico em relação ao conjunto de documentos.

In [7]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=1,
    max_df=0.95
)

print("Vetorizador TF-IDF configurado!")

Vetorizador TF-IDF configurado!


### **Vetorização da Base:**

In [8]:
tfidf_matrix = vectorizer.fit_transform(
    df_knowledge["search_text"]
)

print("========== VETORIZAÇÃO CONCLUÍDA ==========")
print("Documentos:", tfidf_matrix.shape[0])
print("Termos no vocabulário:", tfidf_matrix.shape[1])
print("Formato da matriz:", tfidf_matrix.shape)
print("Tipo da matriz:", type(tfidf_matrix).__name__)

========== VETORIZAÇÃO CONCLUÍDA ==========
Documentos: 47
Termos no vocabulário: 5097
Formato da matriz: (47, 5097)
Tipo da matriz: csr_matrix


### **Identidade e Personalidade da Dália:**

A Dália é uma assistente virtual voltada à saúde e ao acolhimento
de mulheres.

Sua comunicação deve ser humana no tom, sem fingir ser humana,
acolhedora sem ser invasiva e didática sem infantilizar.

#### **Pilares da Personalidade:**
- Acolhimento e empatia.
- Comunicação simples e acessível.
- Respeito à autonomia e às individualidades.
- Curiosidade e escuta ativa.
- Honestidade sobre limitações.
- Responsabilidade e segurança.

#### **Princípios de Comunicação:**
1. Utilizar português brasileiro.
2. Evitar julgamentos, culpabilização e estigmas.
3. Explicar termos técnicos quando necessário.
4. Adaptar a profundidade da explicação ao contexto.
5. Não presumir gênero, orientação sexual, relacionamento ou contexto familiar.
6. Não pressionar a pessoa a tomar decisões.
7. Não inventar informações ou fontes.
8. Reconhecer quando uma questão exige avaliação profissional.

#### **Prioridade de Segurança:**
As regras de segurança e os protocolos de emergência
sempre prevalecem sobre o estilo conversacional.

In [9]:
DALIA_PERSONA = {
    "name": "Dália",
    "language": "pt-BR",

    "personality": [
        "acolhedora",
        "gentil",
        "didática",
        "respeitosa",
        "curiosa",
        "responsável",
        "encorajadora"
    ],

    "communication": {
        "tone": "natural, próximo e acolhedor",
        "complexity": "adaptável ao contexto da conversa",
        "judgment": False,
        "infantilization": False,
        "unnecessary_jargon": False,
        "invented_information": False
    },

    "principles": [
        "respeitar a autonomia da pessoa",
        "não culpabilizar",
        "reconhecer limitações",
        "priorizar informações confiáveis",
        "incentivar acompanhamento profissional quando necessário"
    ],

    "safety": {
        "overrides_personality": True,
        "emergency_priority": True,
        "never_diagnose": True,
        "never_invent_sources": True
    }
}

print("Identidade da Dália configurada!")
print("Pilares:", len(DALIA_PERSONA["personality"]))

Identidade da Dália configurada!
Pilares: 7


### **Função de Recuperação:**

In [10]:
def buscar_conhecimento(pergunta, top_k=5):
    """
    Recupera os documentos mais semelhantes à pergunta.

    Args:
        pergunta (str): pergunta feita à Dália.
        top_k (int): quantidade máxima de documentos retornados.

    Returns:
        pd.DataFrame: documentos ordenados por similaridade.
    """

    if not isinstance(pergunta, str) or not pergunta.strip():
        raise ValueError("A pergunta não pode estar vazia.")

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k deve ser um inteiro maior que zero.")

    # Transformar a pergunta usando o vetorizador já treinado
    vetor_pergunta = vectorizer.transform([pergunta])

    # Calcular similaridade entre pergunta e documentos
    similaridades = cosine_similarity(
        vetor_pergunta,
        tfidf_matrix
    ).flatten()

    # Ordenar os índices pela maior similaridade
    indices = similaridades.argsort()[::-1][:top_k]

    resultados = df_knowledge.iloc[indices][
        ["id", "domain", "title", "urgency", "content", "source_url"]
    ].copy()

    resultados["similaridade"] = similaridades[indices]

    return resultados.reset_index(drop=True)

### **Testes Iniciais:**

In [11]:
pergunta = "Estou grávida e quero saber quando devo começar o pré-natal."

resultados = buscar_conhecimento(pergunta, top_k=5)

display(
    resultados[
        ["id", "title", "urgency", "similaridade"]
    ]
)

,id,title,urgency,similaridade
0,GEST_001,Início e continuidade do pré-natal,rotina,0.135318
1,GEST_005,Diabetes mellitus gestacional,avaliacao,0.107144
2,SEX_003,Infecções sexualmente transmissíveis durante a...,avaliacao,0.080778
3,GEST_003,Risco habitual e alto risco gestacional,informacao_clinica,0.063415
4,GEST_010,Infecção urinária durante a gestação,avaliacao,0.055666


In [13]:
perguntas_teste = [
    "Tenho cólicas menstruais muito fortes.",
    "Quais são os sinais de alerta durante a gravidez?",
    "Estou passando por sofrimento emocional.",
    "Como posso me proteger de uma situação de violência?",
    "Quero informações sobre métodos contraceptivos.",
    "Tenho dúvidas sobre infecções sexualmente transmissíveis."
]

for pergunta in perguntas_teste:
    print("\n" + "=" * 90)
    print("PERGUNTA:", pergunta)

    resultado = buscar_conhecimento(pergunta, top_k=3)

    display(
        resultado[
            ["id", "domain", "title", "urgency", "similaridade"]
        ]
    )


PERGUNTA: Tenho cólicas menstruais muito fortes.


,id,domain,title,urgency,similaridade
0,MEN_003,menstrual,Cólicas menstruais,avaliacao,0.203850
1,MEN_002,menstrual,Quando alterações menstruais merecem avaliação,avaliacao_ou_urgencia_conforme_sintomas,0.111828
2,MEN_007,menstrual,Miomas uterinos,avaliacao,0.059743



PERGUNTA: Quais são os sinais de alerta durante a gravidez?


,id,domain,title,urgency,similaridade
0,GEST_002,gestacional,Sinais de alerta durante a gestação,emergencia,0.155535
1,GEST_006,gestacional,Hipertensão arterial durante a gestação,avaliacao,0.127367
2,GEST_007,gestacional,Pré-eclâmpsia e sinais de alerta,emergencia,0.111448



PERGUNTA: Estou passando por sofrimento emocional.


,id,domain,title,urgency,similaridade
0,MENT_001,saude_mental,Acolhimento diante de sofrimento emocional,acolhimento,0.280009
1,MENT_005,saude_mental,Ansiedade e sofrimento emocional,avaliacao,0.166505
2,MENT_007,saude_mental,Como buscar atendimento em saúde mental no SUS,sempre_aplicavel,0.057544


... (2 outputs left out)


PERGUNTA: Quero informações sobre métodos contraceptivos.


,id,domain,title,urgency,similaridade
0,SEX_002,saude_sexual_reprodutiva,Contracepção e planejamento reprodutivo,educacao,0.253249
1,MENT_003,saude_mental,Resposta segura a pensamentos de suicídio ou a...,emergencia_se_risco_imediato,0.034469
2,SAF_004,seguranca_clinica,"Linguagem inclusiva, autonomia e atendimento s...",governanca,0.025791



PERGUNTA: Tenho dúvidas sobre infecções sexualmente transmissíveis.


,id,domain,title,urgency,similaridade
0,SEX_003,saude_sexual_reprodutiva,Infecções sexualmente transmissíveis durante a...,avaliacao,0.186037
1,SEX_007,saude_sexual_reprodutiva,Prevenção combinada das infecções sexualmente ...,rotina,0.114408
2,GEST_004,gestacional,Puerpério e acompanhamento após o parto,informacao_clinica,0.081433


### **Avaliação Quantitativa:**

Serão avaliados:
- Posição do documento esperado no ranking.
- Precisão da primeira posição.
- Recall@3: presença do documento esperado entre os três primeiros.
- Similaridade do documento recuperado.

Esta avaliação utiliza um conjunto pequeno e controlado de perguntas,
portanto seus resultados são exploratórios e não representam
desempenho clínico ou validação geral do sistema.

In [14]:
casos_teste = [
    {
        "pergunta": "Tenho cólicas menstruais muito fortes.",
        "id_esperado": "MEN_003"
    },
    {
        "pergunta": "Quais são os sinais de alerta durante a gravidez?",
        "id_esperado": "GEST_002"
    },
    {
        "pergunta": "Estou passando por sofrimento emocional.",
        "id_esperado": "MENT_001"
    },
    {
        "pergunta": "Como posso me proteger de uma situação de violência?",
        "id_esperado": "VIO_007"
    },
    {
        "pergunta": "Quero informações sobre métodos contraceptivos.",
        "id_esperado": "SEX_002"
    },
    {
        "pergunta": "Tenho dúvidas sobre infecções sexualmente transmissíveis.",
        "id_esperado": "SEX_003"
    }
]

avaliacoes = []

for caso in casos_teste:
    resultados = buscar_conhecimento(
        caso["pergunta"],
        top_k=len(df_knowledge)
    )

    posicoes = resultados.index[
        resultados["id"] == caso["id_esperado"]
    ].tolist()

    posicao = posicoes[0] + 1 if posicoes else None

    similaridade = (
        resultados.loc[
            resultados["id"] == caso["id_esperado"],
            "similaridade"
        ].iloc[0]
        if posicoes else None
    )

    avaliacoes.append({
        "pergunta": caso["pergunta"],
        "id_esperado": caso["id_esperado"],
        "posicao": posicao,
        "similaridade": similaridade,
        "hit_at_1": posicao == 1,
        "hit_at_3": posicao is not None and posicao <= 3
    })

df_avaliacao = pd.DataFrame(avaliacoes)

display(df_avaliacao)

print("========== MÉTRICAS ==========")
print(
    f"Hit@1: {df_avaliacao['hit_at_1'].mean():.2%}"
)
print(
    f"Hit@3: {df_avaliacao['hit_at_3'].mean():.2%}"
)

,pergunta,id_esperado,posicao,similaridade,hit_at_1,hit_at_3
0,Tenho cólicas menstruais muito fortes.,MEN_003,1,0.203850,True,True
1,Quais são os sinais de alerta durante a gravidez?,GEST_002,1,0.155535,True,True
2,Estou passando por sofrimento emocional.,MENT_001,1,0.280009,True,True
3,Como posso me proteger de uma situação de viol...,VIO_007,1,0.098041,True,True
4,Quero informações sobre métodos contraceptivos.,SEX_002,1,0.253249,True,True
5,Tenho dúvidas sobre infecções sexualmente tran...,SEX_003,1,0.186037,True,True


========== MÉTRICAS ==========
Hit@1: 100.00%
Hit@3: 100.00%


### **Testes de Robustez Linguística:**

Avaliar a recuperação de documentos quando as perguntas são
formuladas com linguagem informal, abreviações, sinônimos
e expressões cotidianas.

O objetivo é verificar se o mecanismo consegue recuperar
conhecimento relevante sem depender da repetição literal
dos termos presentes nos documentos.

In [15]:
perguntas_informais = [
    "Minha menstruação parece uma cachoeira, isso é normal?",
    "Tô grávida e minha cabeça tá doendo muito, devo me preocupar?",
    "Não tô me sentindo bem emocionalmente e queria conversar.",
    "Meu parceiro me ameaça e tenho medo de sair de casa.",
    "Qual é o melhor jeito de evitar uma gravidez?",
    "Acho que posso ter pegado uma IST, e agora?"
]

for pergunta in perguntas_informais:
    print("\n" + "=" * 90)
    print("PERGUNTA:", pergunta)

    resultado = buscar_conhecimento(pergunta, top_k=3)

    display(
        resultado[
            ["id", "domain", "title", "urgency", "similaridade"]
        ]
    )


PERGUNTA: Minha menstruação parece uma cachoeira, isso é normal?


,id,domain,title,urgency,similaridade
0,MEN_001,menstrual,Como funciona o ciclo menstrual,educacao,0.058372
1,GEST_002,gestacional,Sinais de alerta durante a gestação,emergencia,0.056313
2,MEN_002,menstrual,Quando alterações menstruais merecem avaliação,avaliacao_ou_urgencia_conforme_sintomas,0.052935



PERGUNTA: Tô grávida e minha cabeça tá doendo muito, devo me preocupar?


,id,domain,title,urgency,similaridade
0,GEST_011,gestacional,Sinais de alerta no puerpério,emergencia,0.076927
1,GEST_002,gestacional,Sinais de alerta durante a gestação,emergencia,0.062278
2,MENT_003,saude_mental,Resposta segura a pensamentos de suicídio ou a...,emergencia_se_risco_imediato,0.054566



PERGUNTA: Não tô me sentindo bem emocionalmente e queria conversar.


,id,domain,title,urgency,similaridade
0,MENT_003,saude_mental,Resposta segura a pensamentos de suicídio ou a...,emergencia_se_risco_imediato,0.047124
1,MEN_006,menstrual,"Climatério, transição menopausal e menopausa",avaliacao,0.041422
2,VIO_006,violencia,"Violência digital, perseguição e controle tecn...",avaliacao,0.033856


... (2 outputs left out)


PERGUNTA: Qual é o melhor jeito de evitar uma gravidez?


,id,domain,title,urgency,similaridade
0,SEX_006,saude_sexual_reprodutiva,Contracepção de emergência,avaliacao,0.096351
1,MENT_004,saude_mental,Acolhimento em perda gestacional ou neonatal,acolhimento_mais_avaliacao,0.059499
2,MEN_001,menstrual,Como funciona o ciclo menstrual,educacao,0.038258



PERGUNTA: Acho que posso ter pegado uma IST, e agora?


,id,domain,title,urgency,similaridade
0,SEX_005,saude_sexual_reprodutiva,Profilaxia pré-exposição (PrEP) ao HIV,rotina,0.074682
1,VIO_003,violencia,Violência em andamento ou perigo imediato,emergencia,0.060194
2,SAF_004,seguranca_clinica,"Linguagem inclusiva, autonomia e atendimento s...",governanca,0.051701


### **Expansão Lexical de Consultas:**

A expansão lexical acrescenta expressões equivalentes ou
relacionadas à pergunta original, considerando a linguagem
cotidiana utilizada pelas pessoas.

O objetivo é reduzir a dependência de correspondências literais
entre a consulta e os documentos da base.

As expansões são auxiliares e não representam interpretação
clínica ou classificação automática de risco.

In [54]:
EXPANSOES_CONSULTA = {
    "cachoeira": [
    "sangramento menstrual muito intenso",
    "fluxo menstrual excessivo",
    "sangramento uterino anormal",
    "alterações menstruais que merecem avaliação",
    "menstruação com fluxo muito intenso"
    ],

    "cabeça tá doendo muito": [
        "dor de cabeça intensa",
        "cefaleia intensa",
        "sinais de alerta durante a gestação"
    ],

    "não tô me sentindo bem emocionalmente": [
        "sofrimento emocional",
        "acolhimento em saúde mental",
        "apoio emocional"
    ],

    "parceiro me ameaça": [
        "ameaça do parceiro",
        "violência doméstica",
        "violência em andamento",
        "planejamento de segurança"
    ],

    "evitar uma gravidez": [
        "métodos contraceptivos",
        "contracepção",
        "planejamento reprodutivo"
    ],

    "pegado uma ist": [
    "prevenção combinada das infecções sexualmente transmissíveis",
    "suspeita de IST",
    "testagem para IST",
    "diagnóstico e tratamento de IST",
    "procura de atendimento em saúde",
    "infecções sexualmente transmissíveis"
    ],

    "ist": [
    "infecções sexualmente transmissíveis",
    "testagem para IST",
    "diagnóstico e tratamento de IST",
    "prevenção combinada"
    ],

    "infecção sexualmente transmissível": [
        "suspeita de IST",
        "testagem para IST",
        "diagnóstico e tratamento",
        "prevenção combinada"
    ]
}

def expandir_consulta(pergunta):
    consulta = pergunta.lower()
    termos_adicionais = []

    expressoes_encontradas = []

    for expressao, expansoes in EXPANSOES_CONSULTA.items():
        padrao = rf"(?<!\w){re.escape(expressao)}(?!\w)"

        if re.search(padrao, consulta):
            expressoes_encontradas.append(
                (expressao, expansoes)
            )

    # Prioriza expressões mais específicas
    expressoes_encontradas.sort(
        key=lambda item: len(item[0]),
        reverse=True
    )

    for expressao, expansoes in expressoes_encontradas:
        termos_adicionais.extend(expansoes)

    # Remove termos repetidos, preservando a ordem
    termos_adicionais = list(dict.fromkeys(termos_adicionais))

    consulta_expandida = pergunta

    if termos_adicionais:
        consulta_expandida += " " + " ".join(termos_adicionais)

    return consulta_expandida

In [55]:
EXPANSOES_CONSULTA.update({

    "pontos brilhantes": [
        "alterações visuais",
        "visão turva",
        "visão embaçada",
        "sinais de alerta durante a gestação",
        "pré-eclâmpsia"
    ],

    "enxergando pontos": [
        "alterações visuais",
        "visão turva",
        "visão embaçada",
        "sinais de alerta durante a gestação"
    ],

    "tristeza": [
        "sofrimento emocional",
        "acolhimento em saúde mental",
        "apoio emocional"
    ],

    "não sei com quem conversar": [
        "acolhimento diante de sofrimento emocional",
        "escuta respeitosa",
        "apoio emocional"
    ],

    "controla meu celular": [
        "controle de celular",
        "monitoramento",
        "violência digital",
        "controle tecnológico"
    ],

    "controla minhas conversas": [
        "monitoramento de mensagens",
        "controle de celular",
        "violência digital",
        "perseguição"
    ],

    "vacina contra hpv": [
        "vacinação contra o HPV",
        "imunização",
        "papilomavírus humano"
    ],

    "relação sem camisinha": [
        "exposição sexual",
        "relação sexual sem preservativo",
        "profilaxia pós-exposição ao HIV",
        "PEP",
        "contracepção de emergência"
    ],

    "relação desprotegida": [
        "exposição sexual",
        "PEP ao HIV",
        "contracepção de emergência",
        "falha do preservativo"
    ],

    "rompeu a camisinha": [
        "rompimento do preservativo",
        "exposição sexual",
        "PEP ao HIV",
        "contracepção de emergência"
    ]

})

In [56]:
def buscar_conhecimento_expandido(pergunta, top_k=5):
    if not isinstance(pergunta, str) or not pergunta.strip():
        raise ValueError("A pergunta não pode estar vazia.")

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k deve ser um inteiro maior que zero.")

    consulta = expandir_consulta(pergunta)

    vetor_pergunta = vectorizer.transform([consulta])
    similaridades = cosine_similarity(
        vetor_pergunta, tfidf_matrix
    ).flatten()

    indices = similaridades.argsort()[::-1][:top_k]

    resultados = df_knowledge.iloc[indices][
        ["id", "domain", "title", "urgency", "content", "source_url"]
    ].copy()

    resultados["similaridade"] = similaridades[indices]

    return resultados.reset_index(drop=True)

In [57]:
for pergunta in perguntas_informais:

    print("\n" + "=" * 90)
    print("PERGUNTA:", pergunta)

    print("\n--- BUSCA ORIGINAL ---")
    original = buscar_conhecimento(pergunta, top_k=3)

    print(
        original[["id", "title", "similaridade"]]
        .to_string(index=False)
    )

    print("\n--- BUSCA EXPANDIDA ---")
    expandida = buscar_conhecimento_expandido(pergunta, top_k=3)

    print(
        expandida[["id", "title", "similaridade"]]
        .to_string(index=False)
    )


PERGUNTA: Minha menstruação parece uma cachoeira, isso é normal?

--- BUSCA ORIGINAL ---
      id                                          title  similaridade
 MEN_001                Como funciona o ciclo menstrual      0.058372
GEST_002            Sinais de alerta durante a gestação      0.056313
 MEN_002 Quando alterações menstruais merecem avaliação      0.052935

--- BUSCA EXPANDIDA ---
     id                                          title  similaridade
MEN_002 Quando alterações menstruais merecem avaliação      0.284786
MEN_007                                Miomas uterinos      0.182420
MEN_003                             Cólicas menstruais      0.122823

PERGUNTA: Tô grávida e minha cabeça tá doendo muito, devo me preocupar?

--- BUSCA ORIGINAL ---
      id                                                     title  similaridade
GEST_011                             Sinais de alerta no puerpério      0.076927
GEST_002                       Sinais de alerta durante a gestação    

In [58]:
ids_para_inspecionar = [
    "MEN_002",
    "MEN_007",
    "SEX_003",
    "SEX_007",
    "GEST_002",
    "MENT_001"
]

for registro_id in ids_para_inspecionar:
    registro = next(
        item for item in knowledge_base
        if item["id"] == registro_id
    )

    print("\n" + "=" * 90)
    print(f"ID: {registro['id']}")
    print(f"TÍTULO: {registro['title']}")
    print(f"DOMÍNIO: {registro['domain']}")
    print(f"URGÊNCIA: {registro['urgency']}")
    print(f"CONTEÚDO:\n{registro['content']}")
    print(f"FONTE: {registro.get('source_url', '')}")


ID: MEN_002
TÍTULO: Quando alterações menstruais merecem avaliação
DOMÍNIO: menstrual
URGÊNCIA: avaliacao_ou_urgencia_conforme_sintomas
CONTEÚDO:
Sangramento muito intenso, sangramento entre menstruações ou após relação sexual, menstruação que dura mais de sete dias, ciclos persistentemente muito curtos ou longos, ausência menstrual por vários meses, dor incapacitante ou sintomas que prejudicam atividades cotidianas merecem avaliação de saúde. Esses sinais não determinam um diagnóstico específico. Se houver sangramento intenso acompanhado de tontura, desmaio, fraqueza importante ou possibilidade de gravidez com dor/sangramento, orientar atendimento urgente.
FONTE: https://www.acog.org/womens-health/faqs/abnormal-uterine-bleeding

ID: MEN_007
TÍTULO: Miomas uterinos
DOMÍNIO: menstrual
URGÊNCIA: avaliacao
CONTEÚDO:
Miomas uterinos são formações benignas do músculo do útero. Muitas pessoas não apresentam sintomas; em outras, podem ocorrer sangramento menstrual intenso ou prolongado, cóli

### **Avaliar a Qualidade da Recuperação:**

In [59]:
avaliacao_rag = [
    {
        "pergunta": "Minha menstruação parece uma cachoeira, isso é normal?",
        "esperado": "MEN_002"
    },
    {
        "pergunta": "Tô grávida e minha cabeça tá doendo muito, devo me preocupar?",
        "esperado": "GEST_002"
    },
    {
        "pergunta": "Não tô me sentindo bem emocionalmente e queria conversar.",
        "esperado": "MENT_001"
    },
    {
        "pergunta": "Meu parceiro me ameaça e tenho medo de sair de casa.",
        "esperado": "VIO_007"
    },
    {
        "pergunta": "Qual é o melhor jeito de evitar uma gravidez?",
        "esperado": "SEX_002"
    },
    {
        "pergunta": "Acho que posso ter pegado uma IST, e agora?",
        "esperado": "SEX_007"
    }
]

resultados_avaliacao = []

for item in avaliacao_rag:
    resultado = buscar_conhecimento_expandido(
        item["pergunta"],
        top_k=3
    )

    recuperado = resultado.iloc[0]["id"]

    resultados_avaliacao.append({
        "pergunta": item["pergunta"],
        "documento_esperado": item["esperado"],
        "documento_recuperado": recuperado,
        "acertou": recuperado == item["esperado"]
    })

df_avaliacao = pd.DataFrame(resultados_avaliacao)

display(df_avaliacao)

print(
    "Precisão Top-1:",
    round(df_avaliacao["acertou"].mean() * 100, 2),
    "%"
)

,pergunta,documento_esperado,documento_recuperado,acertou
0,"Minha menstruação parece uma cachoeira, isso é...",MEN_002,MEN_002,True
1,"Tô grávida e minha cabeça tá doendo muito, dev...",GEST_002,GEST_002,True
2,Não tô me sentindo bem emocionalmente e queria...,MENT_001,MENT_001,True
3,Meu parceiro me ameaça e tenho medo de sair de...,VIO_007,VIO_007,True
4,Qual é o melhor jeito de evitar uma gravidez?,SEX_002,SEX_002,True
5,"Acho que posso ter pegado uma IST, e agora?",SEX_007,SEX_007,True


Precisão Top-1: 100.0 %


In [60]:
perguntas_reteste = [
    "Descobri que estou grávida e estou enxergando pontos brilhantes.",
    "Tenho sentido muita tristeza e não sei com quem conversar.",
    "Meu namorado controla meu celular e minhas conversas.",
    "Tive uma relação sem camisinha ontem. O que devo fazer?",
    "Quero saber se existe vacina contra HPV."
]

for pergunta in perguntas_reteste:

    print("\n" + "=" * 90)
    print("PERGUNTA:", pergunta)

    print("\nCONSULTA EXPANDIDA:")
    print(expandir_consulta(pergunta))

    print("\nRESULTADOS:")
    resultado = buscar_conhecimento_expandido(
        pergunta,
        top_k=5
    )

    display(
        resultado[["id", "title", "similaridade"]]
    )


PERGUNTA: Descobri que estou grávida e estou enxergando pontos brilhantes.

CONSULTA EXPANDIDA:
Descobri que estou grávida e estou enxergando pontos brilhantes. alterações visuais visão turva visão embaçada sinais de alerta durante a gestação pré-eclâmpsia

RESULTADOS:


,id,title,similaridade
0,GEST_007,Pré-eclâmpsia e sinais de alerta,0.275694
1,GEST_002,Sinais de alerta durante a gestação,0.229318
2,GEST_006,Hipertensão arterial durante a gestação,0.151700
3,GEST_011,Sinais de alerta no puerpério,0.084306
4,GEST_001,Início e continuidade do pré-natal,0.057855



PERGUNTA: Tenho sentido muita tristeza e não sei com quem conversar.

CONSULTA EXPANDIDA:
Tenho sentido muita tristeza e não sei com quem conversar. acolhimento diante de sofrimento emocional escuta respeitosa apoio emocional sofrimento emocional acolhimento em saúde mental

RESULTADOS:


,id,title,similaridade
0,MENT_001,Acolhimento diante de sofrimento emocional,0.398908
1,MENT_007,Como buscar atendimento em saúde mental no SUS,0.149675
2,MENT_005,Ansiedade e sofrimento emocional,0.122623
3,VIO_002,Acolhimento informado por princípios LIVES,0.056094
4,MENT_003,Resposta segura a pensamentos de suicídio ou a...,0.050716



PERGUNTA: Meu namorado controla meu celular e minhas conversas.

CONSULTA EXPANDIDA:
Meu namorado controla meu celular e minhas conversas. controle de celular monitoramento violência digital controle tecnológico

RESULTADOS:


,id,title,similaridade
0,VIO_006,"Violência digital, perseguição e controle tecn...",0.294402
1,VIO_001,Formas de violência contra a mulher,0.045353
2,SAF_002,Privacidade e dados sensíveis,0.036104
3,GEST_005,Diabetes mellitus gestacional,0.034230
4,VIO_003,Violência em andamento ou perigo imediato,0.026362



PERGUNTA: Tive uma relação sem camisinha ontem. O que devo fazer?

CONSULTA EXPANDIDA:
Tive uma relação sem camisinha ontem. O que devo fazer? exposição sexual relação sexual sem preservativo profilaxia pós-exposição ao HIV PEP contracepção de emergência

RESULTADOS:


,id,title,similaridade
0,SEX_004,Profilaxia pós-exposição (PEP) ao HIV,0.399740
1,SEX_006,Contracepção de emergência,0.156371
2,VIO_008,Direito ao atendimento de saúde após violência...,0.128331
3,VIO_005,Atendimento de saúde após violência sexual,0.122578
4,SEX_005,Profilaxia pré-exposição (PrEP) ao HIV,0.109032



PERGUNTA: Quero saber se existe vacina contra HPV.

CONSULTA EXPANDIDA:
Quero saber se existe vacina contra HPV. vacinação contra o HPV imunização papilomavírus humano

RESULTADOS:


,id,title,similaridade
0,SEX_008,Vacinação contra o HPV,0.258778
1,MEN_009,Prevenção do câncer do colo do útero,0.158979
2,GEST_008,Vacinação durante a gestação,0.096478
3,SEX_007,Prevenção combinada das infecções sexualmente ...,0.090405
4,MENT_003,Resposta segura a pensamentos de suicídio ou a...,0.068681


### **Melhorando a Representação dos Documentos:**

In [62]:
# Nova representação dos documentos, com campos ponderados

search_texts_ponderados = []

for registro in knowledge_base:

    titulo = registro.get("title", "")
    topico = registro.get("topic", "")
    palavras_chave = " ".join(registro.get("keywords", []))
    conteudo = registro.get("content", "")

    texto = " ".join([
        titulo, titulo, titulo,
        topico, topico,
        palavras_chave, palavras_chave, palavras_chave,
        conteudo
    ])

    search_texts_ponderados.append(texto)


# Novo vetorizador
vectorizer_ponderado = TfidfVectorizer(
    strip_accents="unicode",
    lowercase=True,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix_ponderada = vectorizer_ponderado.fit_transform(
    search_texts_ponderados
)

print("Documentos:", len(search_texts_ponderados))
print("Vocabulário:", len(vectorizer_ponderado.vocabulary_))
print("Formato da matriz:", tfidf_matrix_ponderada.shape)

Documentos: 47
Vocabulário: 5232
Formato da matriz: (47, 5232)


In [63]:
def buscar_conhecimento_ponderado(pergunta, top_k=5):

    if not isinstance(pergunta, str) or not pergunta.strip():
        raise ValueError("A pergunta não pode estar vazia.")

    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k deve ser um inteiro maior que zero.")

    consulta_expandida = expandir_consulta(pergunta)

    vetor_pergunta = vectorizer_ponderado.transform(
        [consulta_expandida]
    )

    similaridades = cosine_similarity(
        vetor_pergunta,
        tfidf_matrix_ponderada
    ).flatten()

    indices = similaridades.argsort()[::-1][:top_k]

    resultados = df_knowledge.iloc[indices][
        ["id", "domain", "title", "urgency", "content", "source_url"]
    ].copy()

    resultados["similaridade"] = similaridades[indices]

    return resultados.reset_index(drop=True)

In [64]:
perguntas_comparacao = [
    "Estou triste porque terminei um relacionamento.",
    "Meu namorado pediu para olhar meu celular.",
    "Quero aprender sobre HPV.",
    "Estou grávida e tenho dúvidas sobre minha visão.",
    "Quero entender melhor como funciona a contracepção.",
    "Tenho dúvidas sobre saúde mental, mas não estou em crise."
]

for pergunta in perguntas_comparacao:

    print("\n" + "=" * 90)
    print("PERGUNTA:", pergunta)

    print("\n--- TF-IDF ORIGINAL ---")
    original = buscar_conhecimento_expandido(pergunta, top_k=3)
    display(original[["id", "title", "similaridade"]])

    print("\n--- TF-IDF PONDERADO ---")
    ponderado = buscar_conhecimento_ponderado(pergunta, top_k=3)
    display(ponderado[["id", "title", "similaridade"]])


PERGUNTA: Estou triste porque terminei um relacionamento.

--- TF-IDF ORIGINAL ---


,id,title,similaridade
0,VIO_001,Formas de violência contra a mulher,0.074247
1,SAF_001,Limites clínicos e transparência do Dália,0.052355
2,VIO_007,Planejamento de segurança em situações de viol...,0.040441



--- TF-IDF PONDERADO ---


,id,title,similaridade
0,VIO_001,Formas de violência contra a mulher,0.120496
1,SAF_001,Limites clínicos e transparência do Dália,0.042313
2,VIO_007,Planejamento de segurança em situações de viol...,0.033518



PERGUNTA: Meu namorado pediu para olhar meu celular.

--- TF-IDF ORIGINAL ---


,id,title,similaridade
0,VIO_006,"Violência digital, perseguição e controle tecn...",0.074926
1,VIO_008,Direito ao atendimento de saúde após violência...,0.028678
2,SEX_003,Infecções sexualmente transmissíveis durante a...,0.024488


... (14 outputs left out)


PERGUNTA: Tenho dúvidas sobre saúde mental, mas não estou em crise.

--- TF-IDF ORIGINAL ---


,id,title,similaridade
0,MENT_007,Como buscar atendimento em saúde mental no SUS,0.179271
1,MENT_006,Crise de pânico: acolhimento e procura de ajuda,0.103335
2,GEST_004,Puerpério e acompanhamento após o parto,0.098438



--- TF-IDF PONDERADO ---


,id,title,similaridade
0,MENT_007,Como buscar atendimento em saúde mental no SUS,0.173658
1,MENT_006,Crise de pânico: acolhimento e procura de ajuda,0.096181
2,GEST_004,Puerpério e acompanhamento após o parto,0.084113


In [65]:
ids_para_comparar = [
    "MEN_001",
    "SEX_002",
    "SEX_006",
    "MENT_001",
    "VIO_001",
    "VIO_006",
    "GEST_002",
    "GEST_004",
    "GEST_007",
    "MEN_009",
    "SEX_008"
]

for registro_id in ids_para_comparar:

    indice = next(
        i for i, registro in enumerate(knowledge_base)
        if registro["id"] == registro_id
    )

    print("\n" + "=" * 80)
    print("ID:", registro_id)
    print("TEXTO ORIGINAL:")
    print(search_texts[indice])

    print("\nTEXTO PONDERADO:")
    print(search_texts_ponderados[indice])


ID: MEN_001
TEXTO ORIGINAL:


NameError: name 'search_texts' is not defined